# Comparison between the traditional Merge Sort algorithm and a bottom-up CUDA version #
### The following program is made for exercise purposes and does not aim to be fast or optimized. A faster version, using shared memory, will be created later! ###

Created by: Papoulias Dimitrios
* UNIWA email: ice20390183@uniwa.gr
* Gmail: dpapouliask@gmail.com
* GitHub: https://github.com/dimkp

In [ ]:
%%writefile main.cu
#include <iostream>
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <chrono>

#define N 1000000

// Error Checking
static void cudaErrorCheck(cudaError_t e, const char* msg){
    if(e != cudaSuccess){
        printf("CUDA error: %s -> %s\n", msg, cudaGetErrorString(e));
        std::exit(1);
    }
}
// Merge Sort CPU ============================

void merge(int* arr, int left, int mid, int right){
                         
    int n1 = mid - left + 1;
    int n2 = right - mid;

    int* L, * R;
    L = (int*)malloc(n1 * sizeof(int));
    R = (int*)malloc(n2 * sizeof(int));

    for (int i = 0; i < n1; i++)
        L[i] = arr[left + i];
    for (int j = 0; j < n2; j++)
        R[j] = arr[mid + 1 + j];

    int i = 0, j = 0;
    int k = left;
    while (i < n1 && j < n2) {
        if (L[i] <= R[j]) {
            arr[k] = L[i];
            i++;
        }
        else {
            arr[k] = R[j];
            j++;
        }
        k++;
    }
    while (i < n1) {
        arr[k] = L[i];
        i++;
        k++;
    }
    while (j < n2) {
        arr[k] = R[j];
        j++;
        k++;
    }
}

void mergeSort(int* arr, int left, int right){
    if (left >= right)
        return;

    int mid = left + (right - left) / 2;
    mergeSort(arr, left, mid);
    mergeSort(arr, mid + 1, right);
    merge(arr, left, mid, right);
}

// ============================================

// GPU Merge Sort =============================

__device__ void GPUMerge(int* input_arr, int* output_arr, int left, int mid, int right)
{
    int i = left;
    int j = mid;
    int k = left;
    while (i < mid && j < right)
    {
        if (input_arr[i] <= input_arr[j])
        {
            output_arr[k] = input_arr[i];
            i++;
        }
        else
        {
            output_arr[k] = input_arr[j];
            j++;
        }
        k++;
    }

    while (i < mid)
        output_arr[k++] = input_arr[i++];
    while (j < right)
        output_arr[k++] = input_arr[j++];
}

__global__ void mergeKernel(int* input_arr, int* output_arr, int width, int n)
{
    int index = threadIdx.x + blockIdx.x * blockDim.x;
    int left = index * 2 * width;
    if (left >= n)
        return;
    int mid = min(left + width, n);
    int right = min(left + 2 * width, n);
    GPUMerge(input_arr, output_arr, left, mid, right);
}

void GPUMergeSort(int* device_input, int* device_output, int n)
{
    int THREADS_PER_BLOCK = 256;
    for (int width = 1; width < n; width *= 2)
    {
        int BLOCKS_PER_GRID = (n + (2 * width * THREADS_PER_BLOCK) - 1) / (2 * width * THREADS_PER_BLOCK);
        mergeKernel<<<BLOCKS_PER_GRID, THREADS_PER_BLOCK>>>(device_input, device_output, width, n);
        cudaDeviceSynchronize();
        int* temp = device_input;
        device_input = device_output;
        device_output = temp;
    }
}

// ============================================

int main()
{
    int* host_arr;
    int byteSize = N * sizeof(int);
    host_arr = (int*)malloc(byteSize);
    if (!host_arr)
        return 1;
    for (int i = 0; i < N; i++)
        host_arr[i] = N - i;
    for (int i = 0; i < 100; i++)
        printf("%d ", host_arr[i]);
    printf("\n=================================\n");

    // Merge Sort CPU =========================
    auto start = std::chrono::high_resolution_clock::now();
    
    mergeSort(host_arr, 0, N - 1);
    
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;
    std::cout << "CPU Time: " << duration.count() << " ms\n";

    for(int i = 0; i < 100; i++)
        printf("%d ", host_arr[i]);

    // ========================================
    printf("\n===================================\n");
    // Merge Sort GPU =========================

    int* device_arr_input, * device_arr_output;
    cudaErrorCheck(cudaMalloc((void**)&device_arr_input, byteSize), "device_arr malloc");
    cudaErrorCheck(cudaMalloc((void**)&device_arr_output, byteSize), "device_arr_output malloc");
    cudaErrorCheck(cudaMemcpy(device_arr_input, host_arr, byteSize, cudaMemcpyHostToDevice), "host_arr -> device_arr");

    cudaEvent_t d_start, d_stop;
    cudaEventCreate(&d_start);
    cudaEventCreate(&d_stop);
    cudaEventRecord(d_start);

    GPUMergeSort(device_arr_input, device_arr_output, N);
    cudaErrorCheck(cudaMemcpy(host_arr, device_arr_input, byteSize, cudaMemcpyDeviceToHost), "device_arr_output -> host_arr");

    cudaEventRecord(d_stop);
    cudaEventSynchronize(d_stop);
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, d_start, d_stop);
    printf("\nGPU time: %f ms\n", milliseconds);
    
    for(int i = 0; i < 100; i++)
        printf("%d ", host_arr[i]);
    
    // ========================================


    // Memory Cleanup
    free(host_arr);
    cudaErrorCheck(cudaFree(device_arr_input), "free input");
    cudaErrorCheck(cudaFree(device_arr_output), "free output");
    return 0;
}

In [ ]:
!nvcc main.cu -o main

In [ ]:
!./main

## Comments on the results ##

After running multiple tests for different values of N, i consistently see that the GPU implementation is slower than the CPU. But as the N increases, the difference gets smaller but never (at least in my tests) becomes 0 or negative.

That is beacuse: 
-> The GPU is getting utilized at it's full capacity due to me not using shared memory and the **Merge Sort** algorithm not being made for parallel computation without using the proper GPU optimizations.

-> The data transfers between Host and Device, dominate the execution time and increase it significantly in comparison with the CPU Merge Sort, where there are no tranfers.